In [1]:
import os, json, shutil, time, warnings
import numpy as np, deepchem as dc
from rdkit import Chem
from openpom.feat.graph_featurizer import GraphFeaturizer, GraphConvConstants
from openpom.models.mpnn_pom import MPNNPOMModel
warnings.filterwarnings("ignore", category=FutureWarning)

AI  = os.path.expanduser("~/ai-nose")
ENS = f"{AI}/models/ensemble"
APP = f"{AI}/app"; os.makedirs(f"{APP}/models", exist_ok=True)
train = dc.data.DiskDataset(f"{AI}/data/train")
valid = dc.data.DiskDataset(f"{AI}/data/valid")
test  = dc.data.DiskDataset(f"{AI}/data/test")
full  = dc.data.DiskDataset(f"{AI}/data/all")
TASKS = list(train.tasks)

def make_model(model_dir, device="cpu"):
    # Same architecture as in training. No class weights are needed just to make predictions.
    return MPNNPOMModel(
        n_tasks=len(TASKS), batch_size=128, learning_rate=0.001, class_imbalance_ratio=None,
        loss_aggr_type="sum", node_out_feats=100, edge_hidden_feats=75, edge_out_feats=100,
        num_step_message_passing=5, mpnn_residual=True, message_aggregator_type="sum",
        mode="classification", number_atom_features=GraphConvConstants.ATOM_FDIM,
        number_bond_features=GraphConvConstants.BOND_FDIM, n_classes=1,
        readout_type="set2set", num_step_set2set=3, num_layer_set2set=2,
        ffn_hidden_list=[392, 392], ffn_embeddings=256, ffn_activation="relu",
        ffn_dropout_p=0.12, ffn_dropout_at_input_no_act=False, weight_decay=1e-5,
        self_loop=False, optimizer_name="adam", log_frequency=32,
        model_dir=model_dir, device_name=device)

# 1. Copy each model's best checkpoint into the app folder
for i in range(10):
    shutil.copytree(f"{ENS}/model_{i}/best", f"{APP}/models/model_{i}", dirs_exist_ok=True)
json.dump(TASKS, open(f"{APP}/tasks.json", "w"))

# 2. Odor-map position of every catalogue molecule (best model, chosen on validation)
val = {i: json.load(open(f"{ENS}/model_{i}/done.json"))["valid_auroc"] for i in range(10)}
best_i = max(val, key=val.get)
m = make_model(f"{APP}/models/model_{best_i}")
m.restore(model_dir=f"{APP}/models/model_{best_i}")
emb = np.asarray(m.predict_embedding(full), dtype=np.float32).reshape(len(full), -1)

split  = {s: "train" for s in train.ids} | {s: "valid" for s in valid.ids} | {s: "test" for s in test.ids}
smells = [", ".join(TASKS[t] for t in np.where(row == 1)[0]) for row in full.y]
canon  = [Chem.MolToSmiles(Chem.MolFromSmiles(s)) for s in full.ids]
np.savez_compressed(f"{APP}/catalogue.npz", emb=emb,
                    smiles=np.array(full.ids, dtype=str), canon=np.array(canon, dtype=str),
                    smells=np.array(smells, dtype=str),
                    split=np.array([split.get(s, "") for s in full.ids], dtype=str),
                    best_model=best_i)
print("catalogue saved:", emb.shape, "| odor-map model:", best_i)

# 3. Check that the 10-model ensemble works on the CPU, and how fast it is
models = []
for i in range(10):
    mm = make_model(f"{APP}/models/model_{i}"); mm.restore(model_dir=f"{APP}/models/model_{i}"); models.append(mm)
t0 = time.time()
X = dc.data.NumpyDataset(GraphFeaturizer().featurize(["COc1cc(C=O)ccc1O"]))
p = np.mean([np.asarray(mm.predict(X)).reshape(-1) for mm in models], axis=0)
print("vanillin top 5 (CPU):", [TASKS[j] for j in np.argsort(p)[::-1][:5]], f"| {time.time() - t0:.2f}s")
size = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(APP) for f in fs) / 1e6
print(f"app folder size: {size:.0f} MB")

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading material featurizer, missing a dependency. No module named 'ase'
/home/sachinkukkar/miniforge3/envs/open_pom/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading some PyTorch models, missing a dependency. No module

No class imbalance ratio provided!
catalogue saved: (4983, 256) | odor-map model: 6
No class imbalance ratio provided!
No class imbalance ratio provided!
No class imbalance ratio provided!
No class imbalance ratio provided!
No class imbalance ratio provided!
No class imbalance ratio provided!
No class imbalance ratio provided!
No class imbalance ratio provided!
No class imbalance ratio provided!
No class imbalance ratio provided!
vanillin top 5 (CPU): ['vanilla', 'sweet', 'creamy', 'phenolic', 'spicy'] | 0.16s
app folder size: 256 MB


In [1]:
import os, subprocess
from importlib.metadata import version

APP = os.path.expanduser("~/ai-nose/app")

# The exact OpenPOM version (git commit) your models were trained with
commit = subprocess.check_output(["git", "-C", os.path.expanduser("~/openpom"), "rev-parse", "HEAD"], text=True).strip()

# 1. requirements.txt: CPU builds of PyTorch and DGL, plus the exact versions that work on your laptop
pins = ["numpy", "pandas", "scipy", "scikit-learn", "rdkit", "dgllife", "deepchem", "gradio"]
req = ["--extra-index-url https://download.pytorch.org/whl/cpu",
       "--find-links https://data.dgl.ai/wheels/torch-2.4/repo.html",
       "torch==2.4.0+cpu",
       "dgl==2.4.0"]
req += [f"{p}=={version(p)}" for p in pins]
req += [f"openpom @ https://github.com/BioMachineLearning/openpom/archive/{commit}.zip"]
open(f"{APP}/requirements.txt", "w").write("\n".join(req) + "\n")

# 2. Dockerfile: the recipe for building the image
dockerfile = """FROM python:3.10-slim

# System libraries needed by RDKit (molecule drawings) and DGL
RUN apt-get update && apt-get install -y --no-install-recommends libxrender1 libxext6 libgomp1 && rm -rf /var/lib/apt/lists/*

# Run as a normal user with id 1000 (what Hugging Face Spaces expects)
RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user PATH=/home/user/.local/bin:$PATH DGLBACKEND=pytorch PYTHONUNBUFFERED=1 GRADIO_SERVER_NAME=0.0.0.0
WORKDIR /home/user/app

# Install the Python libraries first, so rebuilding after a code change is fast
COPY --chown=user requirements.txt .
RUN pip install --no-cache-dir --user -r requirements.txt

# Then copy the app, the models and the odor-map data
COPY --chown=user . .

EXPOSE 7860
CMD ["python", "app.py"]
"""
open(f"{APP}/Dockerfile", "w").write(dockerfile)

# 3. .dockerignore: files to leave out of the image
open(f"{APP}/.dockerignore", "w").write("__pycache__/\n*.pyc\n.ipynb_checkpoints/\n")

print(open(f"{APP}/requirements.txt").read())
print("Dockerfile and .dockerignore written to", APP)

--extra-index-url https://download.pytorch.org/whl/cpu
--find-links https://data.dgl.ai/wheels/torch-2.4/repo.html
torch==2.4.0+cpu
dgl==2.4.0
numpy==1.26.4
pandas==2.3.3
scipy==1.15.3
scikit-learn==1.7.2
rdkit==2026.9.1
dgllife==0.3.2
deepchem==2.8.1.dev20260821122258
gradio==6.31.0
openpom @ https://github.com/BioMachineLearning/openpom/archive/5434c4b7badebca8e7d180aa34d2f787402121ee.zip

Dockerfile and .dockerignore written to /home/sachinkukkar/ai-nose/app


In [2]:
import os
APP = os.path.expanduser("~/ai-nose/app")

dockerfile = """FROM python:3.10-slim

# System libraries needed by RDKit (molecule drawings) and DGL
RUN apt-get update && apt-get install -y --no-install-recommends libxrender1 libxext6 libgomp1 && rm -rf /var/lib/apt/lists/*

# Run as a normal user with id 1000 (what Hugging Face Spaces expects)
RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user PATH=/home/user/.local/bin:$PATH DGLBACKEND=pytorch PYTHONUNBUFFERED=1 GRADIO_SERVER_NAME=0.0.0.0
WORKDIR /home/user/app

# A newer pip resumes downloads that get cut off, instead of failing
RUN python -m pip install --no-cache-dir --user --upgrade pip

# Install the Python libraries first, so rebuilding after a code change is fast
COPY --chown=user requirements.txt .
RUN python -m pip install --no-cache-dir --user --retries 10 --timeout 120 --resume-retries 10 -r requirements.txt

# Then copy the app, the models and the odor-map data
COPY --chown=user . .

EXPOSE 7860
CMD ["python", "app.py"]
"""
open(f"{APP}/Dockerfile", "w").write(dockerfile)
print("Dockerfile updated")

Dockerfile updated


In [3]:
import os
APP = os.path.expanduser("~/ai-nose/app")

dockerfile = """FROM python:3.10-slim

# System libraries: RDKit's molecule drawings need X render/ext, expat, fontconfig and freetype; DGL needs OpenMP (libgomp1)
RUN apt-get update && apt-get install -y --no-install-recommends libxrender1 libxext6 libexpat1 libfontconfig1 libfreetype6 libgomp1 && rm -rf /var/lib/apt/lists/*

# Run as a normal user with id 1000 (what Hugging Face Spaces expects)
RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user PATH=/home/user/.local/bin:$PATH DGLBACKEND=pytorch PYTHONUNBUFFERED=1 GRADIO_SERVER_NAME=0.0.0.0
WORKDIR /home/user/app

# A newer pip resumes downloads that get cut off, instead of failing
RUN python -m pip install --no-cache-dir --user --upgrade pip

# Install the Python libraries first, so rebuilding after a code change is fast
COPY --chown=user requirements.txt .
RUN python -m pip install --no-cache-dir --user --retries 10 --timeout 120 --resume-retries 10 -r requirements.txt

# Import check: fail the build now if any library can't load, instead of when the app starts
RUN python -c "from rdkit.Chem import Draw; import dgl, deepchem, gradio; from openpom.models.mpnn_pom import MPNNPOMModel; print('all imports OK')"

# Then copy the app, the models and the odor-map data
COPY --chown=user . .

EXPOSE 7860
CMD ["python", "app.py"]
"""
open(f"{APP}/Dockerfile", "w").write(dockerfile)
print("Dockerfile updated")

Dockerfile updated


In [4]:
import os
from huggingface_hub import HfApi

APP = os.path.expanduser("~/ai-nose/app")
api = HfApi()                      # uses the token saved by `hf auth login`
user = api.whoami()["name"]
repo_id = f"{user}/ai-nose"

readme = """---
title: AI Nose
emoji: 👃
colorFrom: indigo
colorTo: green
sdk: docker
app_port: 7860
pinned: false
short_description: Predict how a molecule smells with a GNN ensemble
---

# AI nose: predict how a molecule smells

Type a molecule as a SMILES string. A 10-model graph neural network ensemble (OpenPOM MPNN) predicts
its smell from 138 odor words (test ROC-AUC 0.891), and a learned odor map finds the catalogue
molecules that smell most alike.

- Reproduction of Google Research's Principal Odor Map work (Sanchez-Lengeling et al. 2019; Lee et al., Science 2023)
- Built with OpenPOM, DeepChem, DGL and RDKit; trained on a laptop GPU (RTX 3050, 4 GB)
- Data: GoodScents + Leffingwell odorant catalogues, as curated by OpenPOM (4,983 molecules)

Scores are a ranking, not calibrated probabilities. The model was trained on perfume and flavour
materials, so predictions for very different molecules are less reliable. For research and education.
"""
open(f"{APP}/README.md", "w").write(readme)

api.create_repo(repo_id=repo_id, repo_type="space", space_sdk="docker", exist_ok=True)
api.upload_folder(folder_path=APP, repo_id=repo_id, repo_type="space",
                  ignore_patterns=["__pycache__/*", "*.pyc", ".ipynb_checkpoints/*"],
                  commit_message="AI nose app: 10-model OpenPOM ensemble + odor map")
print("Uploaded. Your Space:", f"https://huggingface.co/spaces/{repo_id}")

HfHubHTTPError: Client error '402 Payment Required' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6ae2f90e-2dcc26c70e6a5aae02a643d4;fb6e8f58-19f5-4c36-bef0-bd6fc60efa21)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

Static Spaces are free for everyone, but hosting Gradio and Docker Spaces on free cpu-basic requires a PRO subscription. Subscribe at https://huggingface.co/pro

In [5]:
import os, glob, json, shutil, tempfile, warnings
import numpy as np, torch, deepchem as dc
from huggingface_hub import HfApi
from openpom.feat.graph_featurizer import GraphFeaturizer, GraphConvConstants
from openpom.models.mpnn_pom import MPNNPOMModel
warnings.filterwarnings("ignore")

AI  = os.path.expanduser("~/ai-nose")
APP = f"{AI}/app"
HUB = f"{AI}/hub"; os.makedirs(f"{HUB}/weights", exist_ok=True)
TASKS = json.load(open(f"{APP}/tasks.json"))

def make_model(model_dir):
    return MPNNPOMModel(
        n_tasks=len(TASKS), batch_size=128, learning_rate=0.001, class_imbalance_ratio=None,
        loss_aggr_type="sum", node_out_feats=100, edge_hidden_feats=75, edge_out_feats=100,
        num_step_message_passing=5, mpnn_residual=True, message_aggregator_type="sum",
        mode="classification", number_atom_features=GraphConvConstants.ATOM_FDIM,
        number_bond_features=GraphConvConstants.BOND_FDIM, n_classes=1,
        readout_type="set2set", num_step_set2set=3, num_layer_set2set=2,
        ffn_hidden_list=[392, 392], ffn_embeddings=256, ffn_activation="relu",
        ffn_dropout_p=0.12, ffn_dropout_at_input_no_act=False, weight_decay=1e-5,
        self_loop=False, optimizer_name="adam", log_frequency=32,
        model_dir=model_dir, device_name="cpu")

# 1. Keep only the weights of each model (drop the optimizer's training state)
for i in range(10):
    ckpt = torch.load(glob.glob(f"{APP}/models/model_{i}/*.pt")[0], map_location="cpu")
    torch.save(ckpt["model_state_dict"], f"{HUB}/weights/model_{i}.pt")
for f in ["catalogue.npz", "tasks.json"]:
    shutil.copy(f"{APP}/{f}", f"{HUB}/{f}")
size = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(HUB) for f in fs) / 1e6
print(f"slimmed files: {size:.0f} MB (was 256 MB)")

# 2. Check: the slimmed weights must give the same predictions as the full checkpoint
X = dc.data.NumpyDataset(GraphFeaturizer().featurize(["COc1cc(C=O)ccc1O"]))
full = make_model(f"{APP}/models/model_0"); full.restore(model_dir=f"{APP}/models/model_0")
slim = make_model(tempfile.mkdtemp()); slim.model.load_state_dict(torch.load(f"{HUB}/weights/model_0.pt"))
print("max prediction difference:", float(np.abs(full.predict(X) - slim.predict(X)).max()))

# 3. Model card: what the model is, how good it is, and its limits
card = """---
license: other
tags: [chemistry, olfaction, odor, graph-neural-network, openpom, deepchem]
---

# AI nose: OpenPOM odor-prediction ensemble

10 message-passing neural networks (OpenPOM `MPNNPOMModel`) that predict 138 odor descriptors
from a molecule's structure, plus a 256-dimensional "odor map" embedding for every catalogue molecule.

## Results (held-out test set, 495 molecules)
- 10-model ensemble: 0.891 mean ROC-AUC over 138 descriptors (95% CI 0.880-0.901)
- Random forest on Morgan fingerprints: 0.862 (ensemble is +0.029 better, 95% CI 0.021-0.036)
- kNN on the learned embedding: 0.818 ROC-AUC vs 0.794 for fingerprints

## Files
- `weights/model_0.pt` ... `model_9.pt`: PyTorch state dicts for OpenPOM `MPNNPOMModel`
  (5 message-passing steps, set2set readout, FFN [392, 392], 256-d embedding, dropout 0.12)
- `catalogue.npz`: odor-map embeddings, SMILES, smell labels and train/valid/test split for 4,983 molecules
- `tasks.json`: the 138 odor descriptors, in model output order

## Training data
GoodScents + Leffingwell odorant catalogues, as curated by OpenPOM (4,983 molecules).
Split by second-order iterative stratification (train 3,978 / valid 510 / test 495).

## Intended use and limitations
For research and education. Trained on perfume and flavour materials, so it is biased toward pleasant
odorants; single molecules only, no odor intensity or mixtures. Scores are a ranking, not calibrated
probabilities. Check OpenPOM's license and the data sources' terms before any commercial use.

## Credits
Reproduction of Sanchez-Lengeling et al. 2019 ("Machine Learning for Scent") and Lee et al., Science 2023
("A principal odor map unifies diverse tasks in olfactory perception"), using OpenPOM, DeepChem, DGL and RDKit.
"""
open(f"{HUB}/README.md", "w").write(card)

# 4. Publish as a free public model repo
api = HfApi()
repo_id = f"{api.whoami()['name']}/ai-nose-openpom"
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=HUB, repo_id=repo_id, repo_type="model",
                  commit_message="10-model OpenPOM odor ensemble (weights only) + odor-map catalogue")
print("Model page:", f"https://huggingface.co/{repo_id}")

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading material featurizer, missing a dependency. No module named 'ase'
/home/sachinkukkar/miniforge3/envs/open_pom/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading some PyTorch models, missing a dependency. No module

slimmed files: 89 MB (was 256 MB)
No class imbalance ratio provided!
No class imbalance ratio provided!
max prediction difference: 0.0
Model page: https://huggingface.co/sachin-kukkar/ai-nose-openpom


In [7]:
import os, subprocess
from importlib.metadata import version
from huggingface_hub import HfApi

AI = os.path.expanduser("~/ai-nose")
repo_id = f"{HfApi().whoami()['name']}/ai-nose-openpom"

app_code = r'''# AI nose: predict how a molecule smells (Streamlit version)
import os, json, tempfile, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("DGLBACKEND", "pytorch")
import numpy as np, pandas as pd, streamlit as st, torch
import deepchem as dc
from huggingface_hub import snapshot_download
from rdkit import Chem
from rdkit.Chem import Draw
from openpom.feat.graph_featurizer import GraphFeaturizer, GraphConvConstants
from openpom.models.mpnn_pom import MPNNPOMModel

HF_REPO = "__HF_REPO__"   # trained weights + odor-map catalogue on the Hugging Face Hub

st.set_page_config(page_title="AI nose", page_icon="👃", layout="wide")

def make_model(n_tasks):
    return MPNNPOMModel(
        n_tasks=n_tasks, batch_size=128, learning_rate=0.001, class_imbalance_ratio=None,
        loss_aggr_type="sum", node_out_feats=100, edge_hidden_feats=75, edge_out_feats=100,
        num_step_message_passing=5, mpnn_residual=True, message_aggregator_type="sum",
        mode="classification", number_atom_features=GraphConvConstants.ATOM_FDIM,
        number_bond_features=GraphConvConstants.BOND_FDIM, n_classes=1,
        readout_type="set2set", num_step_set2set=3, num_layer_set2set=2,
        ffn_hidden_list=[392, 392], ffn_embeddings=256, ffn_activation="relu",
        ffn_dropout_p=0.12, ffn_dropout_at_input_no_act=False, weight_decay=1e-5,
        self_loop=False, optimizer_name="adam", log_frequency=32,
        model_dir=tempfile.mkdtemp(), device_name="cpu")

@st.cache_resource(show_spinner="Loading the 10 models (only on the first visit)...")
def load():
    path = snapshot_download(HF_REPO)
    tasks = json.load(open(os.path.join(path, "tasks.json")))
    cat = dict(np.load(os.path.join(path, "catalogue.npz")))
    models = []
    for i in range(10):
        m = make_model(len(tasks))
        m.model.load_state_dict(torch.load(os.path.join(path, "weights", f"model_{i}.pt"), map_location="cpu"))
        models.append(m)
    emb = cat["emb"] / np.linalg.norm(cat["emb"], axis=1, keepdims=True)
    index = {c: i for i, c in enumerate(cat["canon"])}
    return tasks, cat, emb, index, models, int(cat["best_model"])

WHERE = {"train": "the **training set**: the model has seen it, so this isn't a fair test",
         "valid": "the **validation set**: not trained on, but used to pick the best models",
         "test":  "the **test set**: never used in training, so this is a fair test"}
EXAMPLES = {"Vanillin": "COc1cc(C=O)ccc1O",
            "Isoamyl acetate (banana)": "CC(C)CCOC(C)=O",
            "Limonene (citrus)": "CC1=CCC(CC1)C(=C)C",
            "Ethyl butyrate (fruity)": "CCCC(=O)OCC",
            "Butanal (test set)": "CCCC=O",
            "Aspirin (not a perfume molecule)": "CC(=O)Oc1ccccc1C(=O)O"}

st.title("👃 AI nose: predict how a molecule smells")
st.write("A 10-model graph neural network ensemble (OpenPOM, test ROC-AUC 0.891 on 138 smell words) "
         "predicts how a molecule smells, and a learned odor map finds the catalogue molecules that smell most alike.")

tasks, cat, emb, index, models, best = load()

choice = st.selectbox("Pick an example, or type your own SMILES below", list(EXAMPLES))
smiles = st.text_input("Molecule (SMILES)", value=EXAMPLES[choice]).strip()
mol = Chem.MolFromSmiles(smiles) if smiles else None
if mol is None:
    st.error("Couldn't read that SMILES. Try one of the examples.")
    st.stop()

try:
    X = dc.data.NumpyDataset(GraphFeaturizer().featurize([smiles]))
    p = np.mean([np.asarray(m.predict(X)).reshape(-1) for m in models], axis=0)
    e = np.asarray(models[best].predict_embedding(X)).reshape(-1)
except Exception:
    st.error("The model couldn't process this molecule (it may be too small or unusual).")
    st.stop()

left, right = st.columns(2)
with left:
    st.image(Draw.MolToImage(mol, size=(350, 350)), caption=smiles)
with right:
    top = np.argsort(p)[::-1][:10]
    st.subheader("Top 10 predicted smells")
    st.dataframe(pd.DataFrame({"Smell": [tasks[j] for j in top], "Score": p[top]}), hide_index=True,
                 column_config={"Score": st.column_config.ProgressColumn(min_value=0.0, max_value=1.0, format="%.2f")})

key = Chem.MolToSmiles(mol)
i = index.get(key)
if i is None:
    st.info("**New molecule**: it isn't in the catalogue, so this is a genuine prediction.")
else:
    st.info(f"This molecule is in {WHERE.get(str(cat['split'][i]), 'the catalogue')}.\n\n"
            f"Catalogue smells: **{cat['smells'][i]}**")

sims = emb @ (e / np.linalg.norm(e))
nearest = [k for k in np.argsort(sims)[::-1] if cat["canon"][k] != key][:5]
st.subheader("Closest molecules on the odor map")
st.dataframe(pd.DataFrame({"Molecule": [str(cat["smiles"][k]) for k in nearest],
                           "Its real smells": [str(cat["smells"][k]) for k in nearest],
                           "Similarity": [round(float(sims[k]), 3) for k in nearest]}), hide_index=True)
st.caption("Scores are a ranking, not true probabilities (training gives rare smells extra weight). "
           "Trained on perfume and flavour materials, so predictions for very different molecules are less reliable.")
'''
open(f"{AI}/streamlit_app.py", "w").write(app_code.replace("__HF_REPO__", repo_id))

# requirements.txt for Streamlit Cloud: CPU PyTorch by direct link, DGL from its wheel page, exact versions for the rest
commit = subprocess.check_output(["git", "-C", os.path.expanduser("~/openpom"), "rev-parse", "HEAD"], text=True).strip()
pins = ["numpy", "pandas", "scipy", "scikit-learn", "rdkit", "dgllife", "deepchem", "streamlit", "huggingface_hub"]
req = ["--find-links https://data.dgl.ai/wheels/torch-2.4/repo.html",
       "torch @ https://download-r2.pytorch.org/whl/cpu/torch-2.4.0%2Bcpu-cp310-cp310-linux_x86_64.whl",
       "dgl==2.4.0"]
req += [f"{p}=={version(p)}" for p in pins]
req += [f"openpom @ https://github.com/BioMachineLearning/openpom/archive/{commit}.zip"]
open(f"{AI}/requirements.txt", "w").write("\n".join(req) + "\n")

# packages.txt: system libraries (the same ones the Docker image needed)
open(f"{AI}/packages.txt", "w").write("\n".join(["libxrender1", "libxext6", "libexpat1", "libfontconfig1", "libfreetype6", "libgomp1"]) + "\n")

print("Model repo used by the app:", repo_id)
print(open(f"{AI}/requirements.txt").read())

Model repo used by the app: sachin-kukkar/ai-nose-openpom
--find-links https://data.dgl.ai/wheels/torch-2.4/repo.html
torch @ https://download-r2.pytorch.org/whl/cpu/torch-2.4.0%2Bcpu-cp310-cp310-linux_x86_64.whl
dgl==2.4.0
numpy==1.26.4
pandas==2.3.3
scipy==1.15.3
scikit-learn==1.7.2
rdkit==2026.9.1
dgllife==0.3.2
deepchem==2.8.1.dev20260821122258
streamlit==1.65.0
huggingface_hub==2.3.0
openpom @ https://github.com/BioMachineLearning/openpom/archive/5434c4b7badebca8e7d180aa34d2f787402121ee.zip

